<a href="https://colab.research.google.com/github/DhruvGangwar320/GitHUB/blob/main/ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pygame
import heapq
import random
GRID_SIZE = 15
CELL_SIZE = 50
WINDOW_SIZE = GRID_SIZE * CELL_SIZE
OBSTACLE_COUNT = 10
MOVING_OBSTACLE_COUNT = 4
BUS_STOP_COUNT = 5

# Colors
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
GREEN = (0, 255, 0)
BLUE = (0, 0, 255)
ORANGE = (255, 165, 0)
ROAD_COLOR = (200, 200, 200)
SIDEWALK_COLOR = (230, 230, 230)
pygame.init()
screen = pygame.display.set_mode((WINDOW_SIZE, WINDOW_SIZE + 50))
pygame.display.set_caption("AI Self-Driving Bus Simulation")
clock = pygame.time.Clock()
font = pygame.font.SysFont(None, 24)
bus_img = pygame.image.load("bus.png")
bus_img = pygame.transform.scale(bus_img, (CELL_SIZE-10, CELL_SIZE-10))
bus_stop_img = pygame.image.load("bus_stop.png")
bus_stop_img = pygame.transform.scale(bus_stop_img, (CELL_SIZE-10, CELL_SIZE-10))
car_img = pygame.image.load("car.png")
car_img = pygame.transform.scale(car_img, (CELL_SIZE-10, CELL_SIZE-10))

# --- Helper functions ---
def draw_grid(bus_pos, bus_stops, static_obs, moving_obs, distance_driven, stops_visited, path, road_cells):
    screen.fill(WHITE)
    for y in range(GRID_SIZE):
        for x in range(GRID_SIZE):
            rect = pygame.Rect(x*CELL_SIZE, y*CELL_SIZE, CELL_SIZE, CELL_SIZE)
            if (x, y) in road_cells:
                color = ROAD_COLOR
            else:
                color = SIDEWALK_COLOR
            if (x, y) in static_obs:
                color = BLACK
            pygame.draw.rect(screen, color, rect)
            pygame.draw.rect(screen, BLACK, rect, 1)

    # Draw path
    if path:
        for i in range(len(path)-1):
            start = (path[i][0]*CELL_SIZE + CELL_SIZE//2, path[i][1]*CELL_SIZE + CELL_SIZE//2)
            end = (path[i+1][0]*CELL_SIZE + CELL_SIZE//2, path[i+1][1]*CELL_SIZE + CELL_SIZE//2)
            pygame.draw.line(screen, BLUE, start, end, 3)

    # Draw bus stops
    for stop in bus_stops:
        rect = bus_stop_img.get_rect(center=(stop[0]*CELL_SIZE + CELL_SIZE//2, stop[1]*CELL_SIZE + CELL_SIZE//2))
        screen.blit(bus_stop_img, rect)

    # Draw moving obstacles (cars)
    for car in moving_obs:
        rect = car_img.get_rect(center=(car[0]*CELL_SIZE + CELL_SIZE//2, car[1]*CELL_SIZE + CELL_SIZE//2))
        screen.blit(car_img, rect)

    # Draw bus
    bus_rect = bus_img.get_rect(center=(bus_pos[0]*CELL_SIZE + CELL_SIZE//2, bus_pos[1]*CELL_SIZE + CELL_SIZE//2))
    screen.blit(bus_img, bus_rect)

    # Stats panel
    stats_text = font.render(f"Distance: {distance_driven}   Stops: {stops_visited}/{BUS_STOP_COUNT}", True, BLUE)
    screen.blit(stats_text, (10, WINDOW_SIZE + 10))
    pygame.display.flip()


def heuristic(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def a_star(start, goal, obstacles):
    open_set = []
    heapq.heappush(open_set, (0, start))
    came_from = {}
    g_score = {start:0}
    f_score = {start:heuristic(start, goal)}

    while open_set:
        _, current = heapq.heappop(open_set)
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.reverse()
            return path
        neighbors = [(0,1),(1,0),(0,-1),(-1,0)]
        for dx, dy in neighbors:
            neighbor = (current[0]+dx, current[1]+dy)
            if 0 <= neighbor[0] < GRID_SIZE and 0 <= neighbor[1] < GRID_SIZE:
                if neighbor in obstacles:
                    continue
                tentative_g = g_score[current] + 1
                if neighbor not in g_score or tentative_g < g_score[neighbor]:
                    came_from[neighbor] = current
                    g_score[neighbor] = tentative_g
                    f_score[neighbor] = tentative_g + heuristic(neighbor, goal)
                    heapq.heappush(open_set, (f_score[neighbor], neighbor))
    return []

def move_obstacles(moving_obs, static_obs, bus_pos, bus_stops):
    new_positions = []
    for ox, oy in moving_obs:
        dx, dy = random.choice([(0,1),(1,0),(0,-1),(-1,0),(0,0)])
        nx, ny = ox+dx, oy+dy
        if 0 <= nx < GRID_SIZE and 0 <= ny < GRID_SIZE:
            if (nx, ny) not in static_obs and (nx, ny) not in bus_stops and (nx, ny) != bus_pos:
                new_positions.append((nx, ny))
            else:
                new_positions.append((ox, oy))
        else:
            new_positions.append((ox, oy))
    return new_positions

def nearest_stop(bus_pos, bus_stops):
    if not bus_stops:
        return None
    min_dist = float('inf')
    nearest = None
    for stop in bus_stops:
        dist = heuristic(bus_pos, stop)
        if dist < min_dist:
            min_dist = dist
            nearest = stop
    return nearest

# --- Initialize grid ---
all_positions = [(x, y) for x in range(GRID_SIZE) for y in range(GRID_SIZE)]
static_obstacles = random.sample(all_positions, OBSTACLE_COUNT)
available_positions = [p for p in all_positions if p not in static_obstacles]
moving_obstacles = random.sample(available_positions, MOVING_OBSTACLE_COUNT)
available_positions = [p for p in available_positions if p not in moving_obstacles]
bus_stops = random.sample(available_positions, BUS_STOP_COUNT)

# Define roads randomly (or you can hardcode them for realism)
road_cells = random.sample(available_positions, GRID_SIZE*6)

bus_pos = (0, 0)
distance_driven = 0
stops_visited = 0
current_target = nearest_stop(bus_pos, bus_stops)
obstacles_set = set(static_obstacles + moving_obstacles)
path = a_star(bus_pos, current_target, obstacles_set) if current_target else []

# --- Main loop ---
running = True
while running:
    clock.tick(5)
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    moving_obstacles = move_obstacles(moving_obstacles, static_obstacles, bus_pos, bus_stops)
    obstacles_set = set(static_obstacles + moving_obstacles)

    if not path or any(p in moving_obstacles for p in path):
        if current_target:
            path = a_star(bus_pos, current_target, obstacles_set)

    if path:
        next_pos = path.pop(0)
        distance_driven += 1
        bus_pos = next_pos

    if current_target and bus_pos == current_target:
        if current_target in bus_stops:
            bus_stops.remove(current_target)
            stops_visited += 1
        current_target = nearest_stop(bus_pos, bus_stops)
        if current_target:
            path = a_star(bus_pos, current_target, obstacles_set)
        else:
            path = []

    draw_grid(bus_pos, bus_stops, static_obstacles, moving_obstacles, distance_driven, stops_visited, path, road_cells)

pygame.quit()


In [ ]:
!pip install pygame
!mkdir -p frames

In [ ]:
import pygame
import heapq
import random
import math
import os

# --- Settings ---
GRID_SIZE = 15
CELL_SIZE = 50
WINDOW_SIZE = GRID_SIZE * CELL_SIZE
UI_PANEL_HEIGHT = 50
OBSTACLE_COUNT = 20 # Buildings
MOVING_OBSTACLE_COUNT = 5
BUS_STOP_COUNT = 5

# --- Game-like Settings ---
FPS = 60                       # Run at 60fps for smooth animation
LOGIC_TIMER_MS = 500           # How often logic updates (0.5 seconds)
BUS_MOVE_SPEED = 8.0           # How fast the bus animates (cells per second)
CAR_MOVE_SPEED = 6.0           # How fast the cars animate
BUS_WAIT_TIME = 10             # How many logic "ticks" to wait at a stop

# --- "Cute" Colors ---
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
GREEN = (0, 255, 0)
BLUE = (0, 100, 255)           # Friendlier blue
RED = (255, 0, 0)
GRASS_COLOR = (140, 220, 120)  # Light green for "sidewalk"
ROAD_COLOR = (70, 70, 70)
ROAD_LINE_COLOR = (255, 220, 0) # Yellow
BUILDING_COLOR = (100, 100, 100)
PANEL_COLOR = (50, 150, 255)   # Brighter UI panel

# Initialize pygame
pygame.init()
screen = pygame.display.set_mode((WINDOW_SIZE, WINDOW_SIZE + UI_PANEL_HEIGHT))
pygame.display.set_caption("AI Self-Driving Bus Simulation")
clock = pygame.time.Clock()
font = pygame.font.SysFont(None, 24)
small_font = pygame.font.SysFont(None, 20)

# --- Load Images with Fallbacks ---
def load_image_with_fallback(filename, size, color):
    try:
        image = pygame.image.load(filename)
        image = pygame.transform.scale(image, (size[0] - 10, size[1] - 10))
    except pygame.error:
        print(f"Warning: Could not load image '{filename}'. Using fallback.")
        image = pygame.Surface((size[0] - 10, size[1] - 10))
        image.fill(color)
        image.set_colorkey(BLACK) # Make black transparent if it's a fallback
    return image

bus_img = load_image_with_fallback("bus.png", (CELL_SIZE, CELL_SIZE), GREEN)
bus_stop_img = load_image_with_fallback("bus_stop.png", (CELL_SIZE, CELL_SIZE), BLUE)
car_img = load_image_with_fallback("car.png", (CELL_SIZE, CELL_SIZE), RED)

# --- Helper: Linear Interpolation (Lerp) ---
def lerp(a, b, t):
    """Linearly interpolate from a to b by factor t (0.0 to 1.0)"""
    return a + (b - a) * t

# --- Grid and Road Initialization ---
def create_road_network():
    road_cells = set()
    horizontal_roads = [2, 6, 10, 14]
    vertical_roads = [2, 6, 10, 14]
    for y in range(GRID_SIZE):
        for x in range(GRID_SIZE):
            if x in vertical_roads or y in horizontal_roads:
                road_cells.add((x, y))
    return road_cells

def get_sidewalks(road_cells):
    all_cells = set((x, y) for x in range(GRID_SIZE) for y in range(GRID_SIZE))
    return all_cells - road_cells

def get_stop_locations(road_cells, sidewalk_cells):
    eligible_stops = set()
    for (sx, sy) in sidewalk_cells:
        for dx, dy in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
            if (sx + dx, sy + dy) in road_cells:
                eligible_stops.add((sx, sy))
                break
    return list(eligible_stops)

# --- Drawing Functions ---
def draw_world(bus, bus_stops, static_obs, moving_obs, path, road_cells):
    """Draws the main grid, roads, and entities."""
    screen.fill(GRASS_COLOR) # Default background is grass

    # Draw roads with dashed lines
    for (x, y) in road_cells:
        rect = pygame.Rect(x * CELL_SIZE, y * CELL_SIZE, CELL_SIZE, CELL_SIZE)
        pygame.draw.rect(screen, ROAD_COLOR, rect)

        # Check neighbors to draw dashed lines correctly
        is_horizontal = (x + 1, y) in road_cells or (x - 1, y) in road_cells
        is_vertical = (y + 1, x) in road_cells or (y - 1, x) in road_cells

        if is_horizontal and not is_vertical:
            # Horizontal road
            line_rect = pygame.Rect(x * CELL_SIZE, y * CELL_SIZE + CELL_SIZE//2 - 2, CELL_SIZE, 4)
            pygame.draw.rect(screen, ROAD_LINE_COLOR, line_rect)
        if is_vertical and not is_horizontal:
            # Vertical road
            line_rect = pygame.Rect(x * CELL_SIZE + CELL_SIZE//2 - 2, y * CELL_SIZE, 4, CELL_SIZE)
            pygame.draw.rect(screen, ROAD_LINE_COLOR, line_rect)

    # Draw static obstacles (buildings)
    for (x, y) in static_obs:
        rect = pygame.Rect(x * CELL_SIZE, y * CELL_SIZE, CELL_SIZE, CELL_SIZE)
        pygame.draw.rect(screen, BUILDING_COLOR, rect)

    # Draw bus stops
    for stop in bus_stops:
        rect = bus_stop_img.get_rect(center=(stop[0] * CELL_SIZE + CELL_SIZE // 2, stop[1] * CELL_SIZE + CELL_SIZE // 2))
        screen.blit(bus_stop_img, rect)

    # Draw moving obstacles (cars)
    for car in moving_obs:
        car.draw(screen)

    # Draw bus (with rotation)
    bus.draw(screen)

    # Draw bus status (e.g., "...")
    if bus.state == 'WAITING':
        text = font.render("...", True, BLACK)
        rect = text.get_rect(center=(bus.pixel_pos[0], bus.pixel_pos[1] - CELL_SIZE//2))
        pygame.draw.rect(screen, WHITE, rect.inflate(4, 4))
        pygame.draw.rect(screen, BLACK, rect.inflate(4, 4), 1)
        screen.blit(text, rect)


def draw_ui(distance_driven, stops_visited, current_target):
    panel_rect = pygame.Rect(0, WINDOW_SIZE, WINDOW_SIZE, UI_PANEL_HEIGHT)
    pygame.draw.rect(screen, PANEL_COLOR, panel_rect)
    pygame.draw.rect(screen, BLACK, panel_rect, 2) # Border

    stats_str = f"Distance: {distance_driven}"
    stats_text = font.render(stats_str, True, WHITE)
    screen.blit(stats_text, (10, WINDOW_SIZE + 15))

    stops_str = f"Stops Visited: {stops_visited} / {BUS_STOP_COUNT}"
    stops_text = font.render(stops_str, True, WHITE)
    screen.blit(stops_text, (200, WINDOW_SIZE + 15))

    if current_target:
        target_str = f"Target: {current_target}"
    elif stops_visited == BUS_STOP_COUNT:
        target_str = "Route complete!"
    else:
        target_str = "Target: None"

    target_text = small_font.render(target_str, True, WHITE)
    screen.blit(target_text, (450, WINDOW_SIZE + 18))

# --- Pathfinding (Unchanged) ---
def heuristic(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def a_star(start, goal, obstacles, road_cells):
    open_set = []
    heapq.heappush(open_set, (0, start))
    came_from = {}
    g_score = {start: 0}
    f_score = {start: heuristic(start, goal)}
    while open_set:
        _, current = heapq.heappop(open_set)
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.reverse()
            return path
        neighbors = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        for dx, dy in neighbors:
            neighbor = (current[0] + dx, current[1] + dy)
            if not (0 <= neighbor[0] < GRID_SIZE and 0 <= neighbor[1] < GRID_SIZE):
                continue
            if neighbor not in road_cells or neighbor in obstacles:
                continue
            tentative_g = g_score[current] + 1
            if neighbor not in g_score or tentative_g < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                f_score[neighbor] = tentative_g + heuristic(neighbor, goal)
                heapq.heappush(open_set, (f_score[neighbor], neighbor))
    return []

# --- Animated Entity Class ---
class AnimatedVehicle:
    def __init__(self, image, grid_pos, move_speed):
        self.image = image
        self.grid_pos = grid_pos
        self.pixel_pos = [grid_pos[0] * CELL_SIZE + CELL_SIZE//2, grid_pos[1] * CELL_SIZE + CELL_SIZE//2]
        self.target_pixel_pos = list(self.pixel_pos)
        self.orientation = 0
        self.move_speed = move_speed # In cells per second
        self.state = 'IDLE' # IDLE, MOVING, WAITING
        self.wait_timer = 0

    def update_animation(self, dt):
        """Moves pixel_pos towards target_pixel_pos. dt is delta-time."""
        if self.state != 'MOVING':
            return

        target_x, target_y = self.target_pixel_pos
        current_x, current_y = self.pixel_pos

        # Calculate distance to move this frame
        move_dist = self.move_speed * CELL_SIZE * dt

        # Calculate vector to target
        vec_x, vec_y = target_x - current_x, target_y - current_y
        dist = math.sqrt(vec_x**2 + vec_y**2)

        if dist <= move_dist:
            # Reached target
            self.pixel_pos = list(self.target_pixel_pos)
            self.state = 'IDLE'
        else:
            # Move towards target
            self.pixel_pos[0] += (vec_x / dist) * move_dist
            self.pixel_pos[1] += (vec_y / dist) * move_dist

    def set_move_target(self, next_grid_pos):
        """Sets the next grid cell to move to."""
        self.grid_pos = next_grid_pos
        self.target_pixel_pos = [next_grid_pos[0] * CELL_SIZE + CELL_SIZE//2, next_grid_pos[1] * CELL_SIZE + CELL_SIZE//2]
        self.state = 'MOVING'

        # Update orientation
        dx = self.target_pixel_pos[0] - self.pixel_pos[0]
        dy = self.target_pixel_pos[1] - self.pixel_pos[1]

        if abs(dx) > abs(dy):
            self.orientation = 270 if dx > 0 else 90 # Right, Left
        elif abs(dy) > abs(dx):
            self.orientation = 180 if dy > 0 else 0   # Down, Up

    def draw(self, surface):
        rotated_img = pygame.transform.rotate(self.image, self.orientation)
        rect = rotated_img.get_rect(center=(self.pixel_pos[0], self.pixel_pos[1]))
        surface.blit(rotated_img, rect)

# --- Obstacle Logic ---
def move_obstacles(moving_obs, all_obstacles, road_cells):
    """Logic for moving obstacles (cars)."""
    for car in moving_obs:
        if car.state != 'IDLE':
            continue # Car is still animating

        if random.random() < 0.5: # 50% chance to not move
            continue

        dx, dy = random.choice([(0, 1), (1, 0), (0, -1), (-1, 0)])
        (ox, oy) = car.grid_pos
        nx, ny = ox + dx, oy + dy

        if (nx, ny) in road_cells and (nx, ny) not in all_obstacles:
            car.set_move_target((nx, ny))
            # Update obstacle set for pathfinding
            all_obstacles.remove((ox, oy))
            all_obstacles.add((nx, ny))

def nearest_stop(bus_pos, bus_stops):
    if not bus_stops:
        return None
    return min(bus_stops, key=lambda stop: heuristic(bus_pos, stop))

# --- Initialize Grid and Entities ---
road_cells = create_road_network()
sidewalk_cells = get_sidewalks(road_cells)
static_obstacles = set(random.sample(list(sidewalk_cells), OBSTACLE_COUNT))
eligible_stop_locations = get_stop_locations(road_cells, sidewalk_cells - static_obstacles)
bus_stops = random.sample(eligible_stop_locations, BUS_STOP_COUNT)

available_road_cells = list(road_cells - static_obstacles)
bus_grid_pos = random.choice(available_road_cells)
available_road_cells.remove(bus_grid_pos)

moving_obstacle_list = []
obstacle_grid_positions = set()
for _ in range(MOVING_OBSTACLE_COUNT):
    pos = random.choice(available_road_cells)
    available_road_cells.remove(pos)
    moving_obstacle_list.append(AnimatedVehicle(car_img, pos, CAR_MOVE_SPEED))
    obstacle_grid_positions.add(pos)

# --- Game State Variables ---
bus = AnimatedVehicle(bus_img, bus_grid_pos, BUS_MOVE_SPEED)
distance_driven = 0
stops_visited = 0
current_target_stop = nearest_stop(bus.grid_pos, bus_stops)
all_obstacles = static_obstacles.union(obstacle_grid_positions)
path = a_star(bus.grid_pos, current_target_stop, all_obstacles, road_cells) if current_target_stop else []


# --- Main loop (MODIFIED FOR COLAB) ---

# We will run for a fixed number of frames instead of an infinite loop
TOTAL_FRAMES = 2000
SAVE_EVERY_N_FRAMES = 2 # Set to 1 for 60fps video, 2 for 30fps, etc.

last_logic_update = pygame.time.get_ticks()

print("Starting simulation... This will take a moment.")

for frame_count in range(TOTAL_FRAMES):

    # --- Frame Rate Control ---
    dt = clock.tick(FPS) / 1000.0 # Time in seconds

    # --- Event Handling (Simplified for Colab) ---
    # We just need to pump the event queue, no user input
    pygame.event.pump()

    # --- Logic Update (on a timer) ---
    time_now = pygame.time.get_ticks()
    if time_now - last_logic_update > LOGIC_TIMER_MS:
        last_logic_update = time_now

        # Update obstacle logic
        all_obstacles = static_obstacles.union(set(car.grid_pos for car in moving_obstacle_list))
        move_obstacles(moving_obstacle_list, all_obstacles, road_cells)
        all_obstacles = static_obstacles.union(set(car.grid_pos for car in moving_obstacle_list))

        # Update bus logic
        if bus.state == 'IDLE':
            if any(p in all_obstacles for p in path):
                path = a_star(bus.grid_pos, current_target_stop, all_obstacles, road_cells) if current_target_stop else []

            if path:
                next_pos = path.pop(0)
                bus.set_move_target(next_pos)
                distance_driven += 1

            elif current_target_stop and bus.grid_pos in get_stop_locations(road_cells, set([current_target_stop])):
                bus.state = 'WAITING'
                bus.wait_timer = BUS_WAIT_TIME

        elif bus.state == 'WAITING':
            bus.wait_timer -= 1
            if bus.wait_timer <= 0:
                if current_target_stop in bus_stops:
                    bus_stops.remove(current_target_stop)
                    stops_visited += 1

                current_target_stop = nearest_stop(bus.grid_pos, bus_stops)
                if current_target_stop:
                    path = a_star(bus.grid_pos, current_target_stop, all_obstacles, road_cells)
                else:
                    path = []
                bus.state = 'IDLE'

    # --- Animation Update (every frame) ---
    bus.update_animation(dt)
    for car in moving_obstacle_list:
        car.update_animation(dt)

    # --- Draw ---
    draw_world(bus, bus_stops, static_obstacles, moving_obstacle_list, path, road_cells)
    draw_ui(distance_driven, stops_visited, current_target_stop)
    pygame.display.flip()

    # --- SAVE FRAME (NEW) ---
    if frame_count % SAVE_EVERY_N_FRAMES == 0:
        frame_filename = f"frames/frame_{frame_count//SAVE_EVERY_N_FRAMES:05d}.png"
        pygame.image.save(screen, frame_filename)

    # --- Stop if simulation is done ---
    if stops_visited == BUS_STOP_COUNT and bus.state == 'IDLE' and not path:
        print("Simulation complete!")
        break

pygame.quit()
print(f"Simulation finished. Saved {frame_count // SAVE_EVERY_N_FRAMES} frames.")


In [ ]:
# Starting nlp by processing a text
text = 'Hello World'
print('Original text : ',text)

Original text :  Hello World


In [ ]:
# Lower the text
text = text.lower()
print('Lower character text : ' , text)

Lower character text :  hello world


In [ ]:
# Remove Punctuation
import re
text = re.sub(r'[^\w\s]','',text)
print('Text without punctuation : ',text)

Text without punctuation :  hello world


In [ ]:
sentences = ['I am a happy boy' , 'I am healthy boy','I am a wise boy','I am .....']

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
obj = CountVectorizer()
X = obj.fit_transform(sentences)


In [ ]:
print(obj.get_feature_names_out())

['am' 'boy' 'happy' 'healthy' 'wise']


In [ ]:
print('X : ', X)

X :  <Compressed Sparse Row sparse matrix of dtype 'int64'
	with 10 stored elements and shape (4, 5)>
  Coords	Values
  (0, 0)	1
  (0, 2)	1
  (0, 1)	1
  (1, 0)	1
  (1, 1)	1
  (1, 3)	1
  (2, 0)	1
  (2, 1)	1
  (2, 4)	1
  (3, 0)	1


In [ ]:
print('X,toarray : ',X.toarray())

X,toarray :  [[1 1 1 0 0]
 [1 1 0 1 0]
 [1 1 0 0 1]
 [1 0 0 0 0]]


#TERM FREQUENCE - INVERSE DODOCUMENT FREQUENCY

In [ ]:
# If you are looking at the Class 12 AI or Computer Science curriculum (Natural Language Processing unit), the terms refer to:
# TF (Term Frequency): Measures how often a word appears in a specific document.
# IDF (Inverse Document Frequency): Measures how rare or important a word is across a collection of documents (corpus).
# TF-IDF: A statistical method used to evaluate the importance of a word in a document.


In [ ]:
# tdf = log(no.of docs / term frequency)
# tf = no of term in a docs / total number of words in that document

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
obj2 = TfidfVectorizer()
sentences = "I am happy. Wealthness comes from healthyness. Loves comes from trues or vise versa.".split('.')
Y = obj2.fit_transform(sentences)

In [ ]:
print('Y: ',Y)
print('Y.toarray: ',Y.toarray())


Y:  <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 13 stored elements and shape (4, 11)>
  Coords	Values
  (0, 0)	0.7071067811865476
  (0, 3)	0.7071067811865476
  (1, 10)	0.5552826649411127
  (1, 1)	0.43779123108611473
  (1, 2)	0.43779123108611473
  (1, 4)	0.5552826649411127
  (2, 1)	0.3155366632994909
  (2, 2)	0.3155366632994909
  (2, 5)	0.4002182475169386
  (2, 7)	0.4002182475169386
  (2, 6)	0.4002182475169386
  (2, 9)	0.4002182475169386
  (2, 8)	0.4002182475169386
Y.toarray:  [[0.70710678 0.         0.         0.70710678 0.         0.
  0.         0.         0.         0.         0.        ]
 [0.         0.43779123 0.43779123 0.         0.55528266 0.
  0.         0.         0.         0.         0.55528266]
 [0.         0.31553666 0.31553666 0.         0.         0.40021825
  0.40021825 0.40021825 0.40021825 0.40021825 0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.        ]]


In [ ]:
print('variables : ',obj2.get_feature_names_out())

variables :  ['am' 'comes' 'from' 'happy' 'healthyness' 'loves' 'or' 'trues' 'versa'
 'vise' 'wealthness']


#INTRO TO NLTK LIBRARY

In [ ]:
import nltk

In [ ]:
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
text = 'Hello , I am ready to code'
text = text.lower()

tokens = word_tokenize(text)
print(tokens)

['hello', ',', 'i', 'am', 'ready', 'to', 'code']


In [ ]:
import string
without_punct = [word for word in tokens if word not in string.punctuation]

In [ ]:
rem_stopwords = [t for t in without_punct if t not in stopwords.words('english') ]

In [ ]:
obj = WordNetLemmatizer()
lemm_words = [obj.lemmatize(words) for words in rem_stopwords]


In [ ]:
print('Lemm_words : ',lemm_words)

Lemm_words :  ['hello', 'ready', 'code']


In [ ]:
import numpy as np
x = np.array([12,21,23,43,45])
y = np.array([6,10,12,22,24])
x_mean = np.mean(x)
y_mean = np.mean(y)
slope = np.sum((x-x_mean)*(y-y_mean)) / np.sum((x-x_mean)**2)
intercept = y_mean - slope*x_mean
print('slope : ',slope)
print('intercept : ',intercept)

slope :  0.538534728829686
intercept :  -0.7098001902949562


In [ ]:

y_pred = [slope*z + intercept for z in x]

In [ ]:
print(y)

[ 6 10 12 22 24]


In [ ]:
y_pred

[np.float64(5.752616555661276),
 np.float64(10.59942911512845),
 np.float64(11.676498572787821),
 np.float64(22.44719314938154),
 np.float64(23.524262607040914)]

In [ ]:
import numpy as np

def gradient_descent(X, Y, learning_rate=0.01, epochs=1000):
    m = 0.0  # Initial slope guess
    c = 0.0  # Initial intercept guess
    n = len(X)

    for i in range(epochs):
        # 1. Prediction: y = mx + c
        Y_pred = m * X + c

        # 2. Calculate the "Gradient" (how wrong are we?)
        # These derivatives tell us the direction to the bottom of the "error valley"
        dm = (-2/n) * sum(X * (Y - Y_pred))
        dc = (-2/n) * sum(Y - Y_pred)

        # 3. Update Step: Nudge m and c downhill
        m = m - (learning_rate * dm)
        c = c - (learning_rate * dc)

    return m, c

# Usage with sample data
X = np.array([1, 2, 3, 4, 5])
Y = np.array([2, 4, 5, 4, 5])
m_final, c_final = gradient_descent(X, Y)


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import pandas as pd

# 1. Sample Data
data = {
    'text': ['I love this movie', 'Great film', 'I hated it', 'Terrible acting'],
    'sentiment': [1, 1, 0, 0]  # 1=Positive, 0=Negative
}
df = pd.DataFrame(data)

# 2. Build the Pipeline
# Step 1 ('vect'): Convert text to word count vectors
# Step 2 ('clf'): Predict category based on those counts
text_model = Pipeline([
    ('vect', CountVectorizer()),
    ('clf', LogisticRegression())
])

# 3. Train the entire pipeline at once
text_model.fit(df['text'], df['sentiment'])

# 4. Make a prediction on brand new text
# The pipeline automatically runs the new text through CountVectorizer first!
new_review = ["The movie was great"]
prediction = text_model.predict(new_review)

print(f"Prediction: {'Positive' if prediction[0] == 1 else 'Negative'}")


Prediction: Positive


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer

# 1. Convert your integer data to strings
x = ['1', '2', '2', '3', '4', '5', '6']
y = [2, 4, 6, 7, 8, 9, 9]

pipe = Pipeline([
    # 2. Add 'token_pattern' to ensure single digits aren't ignored
    ('vect', CountVectorizer(token_pattern=r"(?u)\b\w+\b")),
    ('clf', LogisticRegression())
])

# 3. Fit the pipeline
pipe.fit(x, y)

# 4. Pass the prediction input as a list of strings
prediction = pipe.predict(['3'])
print(prediction) # Output: [7]


[7]


#Learn CountVectorizer

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
# does text into numbers
# ngram_range = (min_n, max_n)
obj = CountVectorizer(stop_words = 'english' ,ngram_range = (1
,3))

In [ ]:
text1 = "Brother , how are you ? "
text2 = "I am fine. What about running yourself ?"
grp = obj.fit_transform([text1,text2])

In [ ]:
# Think of it as a librarian who counts words exactly as written
print(obj.get_feature_names_out())
print('grp: ',grp.toarray())

['brother' 'fine' 'fine running' 'running']
grp:  [[1 0 0 0]
 [0 1 1 1]]


#Learn Tf - idf frequency

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
#feature_extraction.text = (CountVectorizer() , TfidfVectorizer() , HashingVectorizer())
obj = TfidfVectorizer()
x = obj.fit_transform(['Hello ther are you fine ', "I am not busy .", "Go and Bye forever."])

In [ ]:
print(x.toarray())

[[0.         0.         0.4472136  0.         0.         0.4472136
  0.         0.         0.4472136  0.         0.4472136  0.4472136 ]
 [0.57735027 0.         0.         0.57735027 0.         0.
  0.         0.         0.         0.57735027 0.         0.        ]
 [0.         0.5        0.         0.         0.5        0.
  0.5        0.5        0.         0.         0.         0.        ]]


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Load dataset
data = pd.read_csv("movie_reviews.csv")

X = data['review']
y = data['sentiment']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Bag of Words
vectorizer = CountVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train model
model = LogisticRegression()
model.fit(X_train_vec, y_train)

# Predictions
y_pred = model.predict(X_test_vec)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# New reviews
new_reviews = [
    "This movie was amazing",
    "Worst movie ever"
]

new_reviews_vec = vectorizer.transform(new_reviews)
predictions = model.predict(new_reviews_vec)

print(predictions)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score
reviews = [
    "This movie was amazing and the acting was brilliant",
    "I really loved the storyline and the characters",
    "The film was fantastic and very entertaining",
    "What a wonderful movie, I would watch it again",
    "The direction and music were excellent",
    "This movie was boring and too long",
    "I hated the acting and the story was terrible",
    "The movie was a complete waste of time",
    "Very bad movie with poor performances",
    "The plot was weak and the film was disappointing"
]

sentiments = [
    "positive","positive","positive","positive","positive",
    "negative","negative","negative","negative","negative"
]
x_train,x_test,y_train,y_test = train_test_split(reviews , sentiments ,test_size = 0.2 , random_state = 42)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder # Import LabelEncoder

obj1 = CountVectorizer()
obj2 = LinearRegression()

# Encode target labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

x_train_vec = obj1.fit_transform(x_train)
x_test_vec = obj1.transform(x_test) # Use transform for test set to avoid re-fitting

obj2.fit(x_train_vec, y_train_encoded) # Fit with encoded labels

LinearRegression()

In [ ]:
prediction = obj2.predict(x_test_vec)
print(prediction)

[0.76609998 0.41637979]


#SVM -Support Vector Machine

In [3]:
# Supervised Machine Learning algorithm for classification and regression
# it tries to sind the boundary ( Hyperbola ) which seperates the different classes of data
# 1. Main Use of SVM

# The primary use of SVM is classification.
# Suppose we have emails labeled:

# Spam

# Not Spam
# wx + b = y
# This equation represents the decision boundary separating classes.

# SVM chooses the boundary that maximizes the margin (distance between the line and the nearest data points).

# Those closest points are called support vectors.

In [ ]:
# 2. Real-World Uses of SVM
# 1. Text Classification (NLP)

# Used in:

# Sentiment analysis

# Spam detection

# News categorization

# Example:
# Movie review → Positive or Negative 🎬

# 2. Image Classification

# SVM can classify images such as:

# Cat vs Dog 🐱🐶

# Face recognition

# Object detection

# 3. Bioinformatics

# Used for:

# Gene classification

# Cancer detection from medical data 🧬

# 4. Handwriting Recognition

# Example:

# Recognizing digits in postal codes or bank cheques ✍️

# 5. Fraud Detection

# Banks use SVM to detect:

# Fraudulent transactions 💳

# 3. Types of SVM

# There are mainly two types:

# 1. Linear SVM

# Used when data can be separated by a straight line.

# 2. Non-Linear SVM

# When data is messy and tangled, SVM uses kernels to transform data to higher dimensions.

# Common kernels:

# Linear kernel

# Polynomial kernel

# RBF (Gaussian) kernel

# 4. Why SVM is Powerful

# SVM works well when:

# Dataset is high dimensional

# Number of features > number of samples

# Need clear classification boundary

# ✔ In one sentence:
# SVM is used to find the optimal boundary that separates different classes of data with maximum margin.